# Viz — Plotly + Kaleido

Interactive scatter plus static ``write_image`` (needs ``kaleido``).

**Interactive WebGL scatter**

Builds a clipped subsample in stereo north coordinates and opens an interactive Plotly figure (`webgl` for speed) on a **1400×1400** layout. Pan/zoom to inspect dense regions; color encodes `log1p(degree)` like the matplotlib notebook.

In [ ]:
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import dmercator_io as dm

merged = dm.load_merged_parquet(Path("cache") / "merged.parquet")
u, v = dm.stereo_disk_north(merged)
clip = 9.0
m = (np.abs(u) < clip) & (np.abs(v) < clip)
sub = merged.loc[m].copy()
sub["u"] = u[m]
sub["v"] = v[m]

fig = px.scatter(
    sub,
    x="u",
    y="v",
    color=np.log1p(sub["degree"].to_numpy()),
    render_mode="webgl",
    labels={"color": "log1p(deg)"},
    title="Stereo north (subsample/clipped for speed)",
)
fig.update_traces(marker=dict(size=3, opacity=0.35))
fig.update_layout(width=1400, height=1400)  # larger canvas ≈ 2× typical default
fig.show()


**Static image export (Kaleido)**

Writes a standalone PNG via Plotly’s Kaleido engine at **1800×1800** pixels (`scale=2` unchanged). Run the scatter cell above first so `fig` exists. **Interpretation:** this is a snapshot of the clipped view, not the full dynamic range of the interactive plot.

In [ ]:
out = Path("cache") / "figures"
out.mkdir(parents=True, exist_ok=True)
fig.write_image(out / "plotly_stereo.png", width=1800, height=1800, scale=2)
print("Wrote", (out / "plotly_stereo.png").resolve())
